# Permutation-preserving Jaccard Distance (P-Jaccard) 


### ------------------------------------------------------------------------
### Purpose: Generate p-values for Del-Dup burden correlations
### Author : Sayeh Kazem
### Last Updated: 2025-08-05
### Python Version: 3.10
### ------------------------------------------------------------------------

### Script Overview
The current version of the script calculates P-Jaccard for the Del-Dup burden correlations for each trait. It does this by generating 1,000 Jaccard-preserving null values.

### Required Input Data
Please ensure the following input files are available:
- Gene Set Files: The script requires gene set files to be provided in the Gene_Sets/ directory. Each file must be a tab-separated values (.tsv) file containing a single column labeled gene_list.
- Effect Size Files: The script requires the Del and Dup effect size files to be in a long format and placed in the EffectSizes/ directory as a .csv file with the  Geneset, Trait, Effectsize and CNV_TYPE Columns; the script will reformat the effect sizes into two separate matrices, one for deletions and one for duplications. In both matrices, the gene sets will be on the rows and traits will be on the columns. The script will automatically ensure the gene set order is consistent between genesets and effectsize files.

The script generates two p-values:
- Fixing Del and surrogating Dup profiles.
- Fixing Dup and surrogating Del profiles.
You can use the P-Jaccards that are either sig by Delbased or Dupbased method. Then you can do FDR correction.


### Required Output Directories

Make sure to create/have these folders in the script paths:  
- Gene_Sets/
- EffectSizes/
- JaccardDist/
- surrogate_maps_DelDup/
- PJaccard/

## Functions

In [1]:
### Functions ######
# Import Packages
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import nibabel
import brainsmash
from scipy.spatial import distance
from scipy import stats
from statsmodels.stats.multitest import multipletests
import itertools
import glob
import os
from brainsmash.mapgen.base import Base
import csv
from brainsmash.mapgen.stats import pearsonr
from brainsmash.mapgen.stats import nonparp
from concurrent.futures import ProcessPoolExecutor
import random
from brainsmash.mapgen.eval import base_fit
import time


## Dist Matrix
def read_gene_list(file_path):
    df = pd.read_csv(file_path, sep='\t', header=None, skiprows=1)
    return set(df[0])

def jaccard_distance(set1, set2):
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return 1.0 - intersection / union

def compute_jaccard_distance_matrix(folder_path):

    file_paths = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.tsv')]

    gene_lists = [read_gene_list(file_path) for file_path in file_paths]
    
    num_files = len(file_paths)

    # Initialize a square matrix to store distances
    distance_matrix = [[0.0] * num_files for _ in range(num_files)]

    # Compute Jaccard distance between all pairs of gene lists
    for i, j in itertools.combinations(range(num_files), 2):
        dist = jaccard_distance(gene_lists[i], gene_lists[j])
        distance_matrix[i][j] = dist
        distance_matrix[j][i] = dist

    return distance_matrix
    #print(file_paths)

def save_distance_matrix_to_csv(distance_matrix, file_path):
    df = pd.DataFrame(distance_matrix)
    df.to_csv(file_path, sep='\t', index=False, header=False)
    

def process_phenotype_files(file):
    #data_del = np.loadtxt(del_file)
    data = np.loadtxt(file)
    return data

def surrogate_analysis(phenotype_DEL, phenotype_DUP,
                      trait_x):
    # Fantom ###############################
    base_DEL = Base(x=phenotype_DEL, D='JaccardDist/JaccardDist_Matrix.txt')
    base_DUP = Base(x=phenotype_DUP, D='JaccardDist/JaccardDist_Matrix.txt')

    
    surrogate_maps_DEL = base_DEL(n=1000)
    surrogate_maps_DUP = base_DUP(n=1000)

    # Create directory for saving maps
    os.makedirs('surrogate_maps_DelDup', exist_ok=True)

    # Save each map individually
    for i, map_del in enumerate(surrogate_maps_DEL):
        np.save(f'surrogate_maps_DelDup/surrogate_map_DEL_{trait_x}_{i}.npy', map_del)
        
    for i, map_dup in enumerate(surrogate_maps_DUP):
        np.save(f'surrogate_maps_DelDup/surrogate_map_DUP_{trait_x}_{i}.npy', map_dup)
    
    fixed_DEL = process_phenotype_files(phenotype_DEL)
    fixed_DUP = process_phenotype_files(phenotype_DUP)

    
    ########################################
    ### Pearson Corr Computation 
    ########################################
    
    surrogate_corrs_1 = pearsonr(fixed_DUP, surrogate_maps_DEL).flatten()
    surrogate_corrs_2 = pearsonr(fixed_DEL, surrogate_maps_DUP).flatten()

    # print("Surrogate Brainmap Correlations:", surrogate_brainmap_corrs)

    test_stat = pearsonr(fixed_DEL, fixed_DUP)[0]
    # print("Test Statistic:", test_stat)

    surrogate_p_value_1 = nonparp(test_stat, surrogate_corrs_1)
    surrogate_p_value_2 = nonparp(test_stat, surrogate_corrs_2)
    # print("Surrogate P-Value:", surrogate_p_value)

    return surrogate_p_value_1 , surrogate_p_value_2

def surrogate_analysis_parallel(pair):
    x = pair
    del_file = f"EffectSizes/DEL_{x}.txt"
    dup_file = f"EffectSizes/DUP_{x}.txt"
    print(x)
    surrogate_p_value_1, surrogate_p_value_2 = surrogate_analysis(del_file, dup_file,x)
    return {'trait1': x, 'surrogate_p_value_delBased': surrogate_p_value_1,'surrogate_p_value_dupBased': surrogate_p_value_2}



## Jaccard Index 

In [2]:
#A1 : Read Gene-lists + Jaccard Index


# Read Tissue Geneset

folder_path = 'Gene_Sets/'
distance_matrix = compute_jaccard_distance_matrix(folder_path)

# Specify the file path to save the distance matrix
output_file_path = 'JaccardDist/JaccardDist_Matrix.txt'

# Save the distance matrix to a delimited text file
save_distance_matrix_to_csv(distance_matrix, output_file_path)

# Get a list of all files in the folder
files = os.listdir(folder_path)

# Filter TSV files
tsv_files = [file for file in files if file.endswith('.tsv')]
tsv_files = [file.replace('.tsv', '') for file in tsv_files]
# Print the names of TSV files
#for tsv_files in tsv_files:
 #   print(tsv_files)


### Jaccard txt
process_phenotype_files(output_file_path)

array([[0.        , 0.93044822, 0.49756362, 0.45939933, 0.99663551,
        0.99125364, 0.99680851, 0.99074759, 0.99590011, 0.99200872],
       [0.93044822, 0.        , 0.94236641, 0.93451464, 0.9829222 ,
        0.98494308, 0.98271516, 0.93881528, 0.98452246, 0.96022514],
       [0.49756362, 0.94236641, 0.        , 0.28509586, 0.99476244,
        0.98905509, 0.9968119 , 0.99038462, 0.99515648, 0.99311345],
       [0.45939933, 0.93451464, 0.28509586, 0.        , 0.99475262,
        0.99161502, 0.99716211, 0.99148779, 0.9977662 , 0.9949257 ],
       [0.99663551, 0.9829222 , 0.99476244, 0.99475262, 0.        ,
        0.94321767, 0.98217601, 0.92723577, 0.94834544, 0.82200789],
       [0.99125364, 0.98494308, 0.98905509, 0.99161502, 0.94321767,
        0.        , 0.97940751, 0.95067437, 0.8697479 , 0.9387208 ],
       [0.99680851, 0.98271516, 0.9968119 , 0.99716211, 0.98217601,
        0.97940751, 0.        , 0.96927064, 0.97197309, 0.96339678],
       [0.99074759, 0.93881528, 0.9903846

In [3]:
tsv_files

['medial_temporal_gyrus',
 'pituitary_gland',
 'insular_cortex',
 'frontal_lobe',
 'liver',
 'heart_muscle',
 'appendix',
 'thyroid_gland',
 'breast',
 'kidney']

## EffectSizes

In [4]:

## B1) Constructing effectsizes txt files

df_effectsize = pd.read_csv("EffectSizes/df_funburd_ukbb.csv" )
traits_to_keep = ["BMI", "FI", "EA", "LDL", "HeartRate"]
df_effectsize = df_effectsize[df_effectsize['Trait'].isin(traits_to_keep)]

# For DEL
df_effectsize_del=df_effectsize[df_effectsize.CNV_Type=='DEL']
df_effectsize_del
# For DUP
df_effectsize_dup=df_effectsize[df_effectsize.CNV_Type=='DUP']
df_effectsize_dup


##### Change the format to the matrix format #### (Geneset*Traits)#######

matrix_effectsize_dup = df_effectsize_dup.pivot(index='Geneset', columns='Trait', values='Effectsize')
matrix_effectsize_del = df_effectsize_del.pivot(index='Geneset', columns='Trait', values='Effectsize')



## ordering effectsize df by genesets + Create txt files###
order = tsv_files
matrix_effectsize_del_order = matrix_effectsize_del.loc[order]
#effectsize_DEL_file.head()

for column in matrix_effectsize_del_order.columns:
    column_values = matrix_effectsize_del_order[column]
    file_name = f"EffectSizes/DEL_{column}.txt"
    
    # Save only the values without index or column names
    column_values.values.tofile(file_name, sep='\n')


matrix_effectsize_dup_order = matrix_effectsize_dup.loc[order]
#effectsize_DUP_file.head()

for column in matrix_effectsize_dup_order.columns:
    column_values = matrix_effectsize_dup_order[column]
    file_name = f"EffectSizes/DUP_{column}.txt"
    
    # Save only the values without index or column names
    column_values.values.tofile(file_name, sep='\n')
    print(column)
################################################################

matrix_effectsize_dup_order.head(5)

BMI
EA
FI
HeartRate
LDL


Trait,BMI,EA,FI,HeartRate,LDL
Geneset,,,,,
medial_temporal_gyrus,0.010086,-0.042935,-0.079222,0.021482,0.002765
pituitary_gland,-0.009369,-0.005227,0.003195,0.015171,-0.012442
insular_cortex,0.027431,-0.037158,-0.062741,0.016264,0.002752
frontal_lobe,0.025828,-0.049838,-0.078365,0.013164,0.000186
liver,0.016109,-0.003486,0.001182,-0.007290,0.006491


## Surrogate Analysis & P-value computation (On subset of genesets and Traits)

In [5]:
## This Cell Takes Time to Run!!!

In [6]:
##Part C : Serrogate maps for DEL and Dup Seperatley 
random.seed(43)
X = matrix_effectsize_dup_order.columns

# Generate pairs for parallel processing
pairs = [(x) for i, x in enumerate(X)]
        
# Use ThreadPoolExecutor for parallel processing
with ProcessPoolExecutor(max_workers=6) as executor:
    results_list = list(executor.map(surrogate_analysis_parallel, pairs))
# Create the DataFrame from the list of results
result_df = pd.DataFrame(results_list)
result_df.head()

HeartRateBMILDLFIEA






,trait1,surrogate_p_value_delBased,surrogate_p_value_dupBased
0,BMI,0.557,0.654
1,EA,0.000,0.008
2,FI,0.581,0.464
3,HeartRate,0.118,0.094
4,LDL,0.187,0.311


In [7]:
result_df.to_csv('PJaccard/PJaccard.csv', index=False)

## FDR P-Jaccard 

In [8]:
_, pvalues_corrected_delBased, _, _ = multipletests(result_df.surrogate_p_value_delBased, method='fdr_bh')
pvalues_corrected_delBased_dic=dict(zip(matrix_effectsize_del_order.columns, pvalues_corrected_delBased))
pvalues_corrected_delBased_dic

{'BMI': 0.581,
 'EA': 0.0,
 'FI': 0.581,
 'HeartRate': 0.295,
 'LDL': 0.3116666666666667}

In [9]:
_, pvalues_corrected_dupBased, _, _ = multipletests(result_df.surrogate_p_value_dupBased, method='fdr_bh')
pvalues_corrected_dupBased_dic=dict(zip(matrix_effectsize_dup_order.columns, pvalues_corrected_dupBased))
pvalues_corrected_dupBased_dic

{'BMI': 0.654,
 'EA': 0.04,
 'FI': 0.58,
 'HeartRate': 0.235,
 'LDL': 0.5183333333333333}

In [10]:
pvalues_corrected_delBased_df = pd.DataFrame(list(pvalues_corrected_delBased_dic.items()), columns=['Trait', 'FDR_PJaccard'])
pvalues_corrected_dupBased_df = pd.DataFrame(list(pvalues_corrected_dupBased_dic.items()), columns=['Trait', 'FDR_PJaccard'])

pvalues_corrected_delBased_df.to_csv('PJaccard/PJaccard_FDR_delBased.csv', index=False)
pvalues_corrected_dupBased_df.to_csv('PJaccard/PJaccard_FDR_dupBased.csv', index=False)

## Either Condition ##
X1=pvalues_corrected_dupBased_df.Trait[pvalues_corrected_dupBased_df.iloc[:,1]<0.05] 
X2=pvalues_corrected_delBased_df.Trait[pvalues_corrected_delBased_df.iloc[:,1]<0.05] 
